# Imports

In [ ]:
import re
import cda2
import math

import time
from datetime import datetime, timedelta

import pandas as pd
import pyarrow
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf

from pyspark.sql.window import Window
import pyspark.sql.types as T
import pyspark.sql.functions as F
from pyspark.sql.functions import col, row_number
from pyspark.sql.functions import explode, map_keys, col

In [ ]:
api = cda2.Api()

In [ ]:
# Set configuration parameters to better optimize queries.

config = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.enabled": "true",
    "spark.sql.adaptive.coalescePartitions.parallelismFirst": "false",
    "spark.sql.adaptive.coalescePartitions.minPartitionSize": "1m",
    "spark.executor.memory": "8g",
    "spark.executor.memoryOverhead": "16g",
}

In [ ]:
# Start Spark and specify number of cpus to use.

api.start_spark(n_executors=100, config=config)

In [ ]:
%run ./shared_variables.ipynb

In [ ]:
#year0 = "2025"
year1 = str(int(year0) + 1)

In [ ]:
dates = {"start_date": year0 + "-01-01", "end_date": year1 +"-01-01"}

# retrieve fav

In [ ]:
df_airspaces = (
    api.dataframe("StaticEramAirspace",
                  **dates,
#                  partition_filters=api.custom_partitions(['ARTCC_SECTOR']), // additional: {FULLNAME -> SECT1} 
#                  partition_filters=api.custom_partitions(['SECTOR']), // additional: {ACTIVE -> false} or {ACTIVE -> true}
#                  partition_filters=api.custom_partitions(['TRACON']), // additional: {}
#                  partition_filters=api.custom_partitions(['SAA']), // additional: {ACTIVE -> ALWAYS_OFF, TYPE -> LOCAL_ASSIGNED}
#                  partition_filters=api.custom_partitions(['FAV']), # additional: {TYPE -> ENROUTE} ; {TYPE -> APPROACH, ARTSID -> MCI} ; 
#                  partition_filters=api.custom_partitions(['ZDC']),
                  partition_filters=api.custom_partitions(['FAV']),
                  metadata=True)
    .select (
        "center",
        "type",
        "identifier",
        "additional",
        F.explode('geometry.modules').alias('module'),
        "chart_date"
    )
    .withColumnRenamed('center', 'artcc')
    .withColumnRenamed('identifier', 'fav')
    .withColumn("grouping", F.concat("artcc", F.lit("_"), "fav", F.lit("_"), "module.identifier"))
    .orderBy("grouping")
)

In [ ]:
#window = Window.partitionBy("grouping").orderBy(col("end_date").desc())
#window = Window.partitionBy("grouping").orderBy(col("metadata.effective_end_date").desc())
window = Window.partitionBy("grouping").orderBy(col("chart_date").desc())

df_airspaces_newest = (df_airspaces
    .withColumn("row", row_number().over(window))
    .filter(col("row") == 1)
    .drop("row", "chart_date")
    .orderBy("grouping")
)

In [ ]:
df_airspaces_newest = (df_airspaces_newest
    .withColumn("fav_type", F.concat("type", F.lit('-'), df_airspaces["additional"].getItem("TYPE")))
#    .withColumn("fav_type", df_airspaces["additional"].getItem("TYPE"))
#    .withColumn("tracon", df_airspaces["additional"].getItem("ARTSID"))
    .withColumn("tracon", F.coalesce(df_airspaces["additional"].getItem("ARTSID"), F.lit("")))
    .withColumn("approach_control_id", F.coalesce(df_airspaces["additional"].getItem("APPROACH_CONTROL_ID"), F.lit("")))
    .drop("type")
    .drop("additional")
    .withColumnRenamed('fav_type', 'type')
)

In [ ]:
#df_airspaces_newest.count()

In [ ]:
#df_airspaces_newest.show()

## keep only the newest update

In [ ]:
#df_airspaces_newest.count()

In [ ]:
#df_airspaces_newest.show()

In [ ]:
module_schema = T.StructType([
    T.StructField("module", T.StructType([
        T.StructField("identifier", T.DoubleType(), True),
        T.StructField("floor", T.DoubleType(), True),
        T.StructField("ceiling", T.DoubleType(), True),
        T.StructField("polygon", T.StructType([
            T.StructField("boundary", T.ArrayType(
                T.StructType([
                    T.StructField("latitude", T.DoubleType(), True),
                    T.StructField("longitude", T.DoubleType(), True),
                    T.StructField("sequence_number", T.IntegerType(), True),
                ]), True), True),
        ]), True),
    ]), True),
])

In [ ]:
none_string = "*"
separator_string = ":"

#@F.udf(T.StringType())
@F.udf(T.MapType(T.StringType(),T.StringType()))
def extract_boundary_from_module(module:module_schema) -> map:  
    result = {}

    result["shelf"] = f'{module.identifier}'
    result["floor"] = f'{module.floor:.0f}'
    result["ceiling"] = f'{module.ceiling:.0f}'

    boundary_string = ""

    module.polygon.boundary.sort(key=lambda x: x.sequence_number, reverse=False)

    for point in module.polygon.boundary:
        boundary_string += (none_string if point.latitude is None else f'{point.latitude:.6f}') + " "
        boundary_string += (none_string if point.longitude is None else f'{point.longitude:.6f}') + separator_string

    # drop the trailing separator
    if len(boundary_string) > 0:
        boundary_string = boundary_string[0:len(separator_string) * -1]

    result["boundary"] = boundary_string
    result["shelf"] = module.identifier

    return result

In [ ]:
df_airspaces_dict = (
    df_airspaces_newest
    .withColumn("module_dict", extract_boundary_from_module("module"))
    .select(
        "artcc",
        "type",
        "fav",
        "tracon",
        "approach_control_id",
        "module_dict",
    )
    .drop("module")
    .orderBy("artcc", "type", "fav")
)

In [ ]:
#df_tracons_dict.show(1, truncate=False)

In [ ]:
# expand the dictionary to columns

# https://mungingdata.com/pyspark/dict-map-to-multiple-columns/
# https://stackoverflow.com/questions/36869134/pyspark-converting-a-column-of-type-map-to-multiple-columns-in-a-dataframe

keys = ["shelf", "ceiling", "floor", "boundary"]
key_cols = list(map(lambda f: F.col("module_dict").getItem(f).alias(str(f)), keys))
final_cols = [
        "artcc",
        "type",
        "fav",
        "tracon",
        "approach_control_id",
             ] + key_cols

In [ ]:
df_airspaces_output = (
    df_airspaces_dict.select(final_cols)
    .withColumnRenamed('boundary', 'geometry')
)

In [ ]:
#df_airspaces_output.show(10)

In [ ]:
(
    df_airspaces_output
    .write.option("header", True)
    .csv("CRAFT/" + year0 + "/airspaces/favs", compression="None", mode="overwrite")
)